# Import Libraries

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import pearsonr
import seaborn as sns
import yaml
from typing import List
from rdkit import Chem

repo_path = os.path.dirname(os.path.abspath(""))
CHECKOUT_PATH = repo_path
DATASET_PATH = os.path.join(repo_path, "datasets")

os.chdir(CHECKOUT_PATH)
sys.path.insert(0, CHECKOUT_PATH)

In [ ]:
# Setting up plotting parameters

matplotlib.rcParams.update(
    {
        "pgf.texsystem": "pdflatex",
        "font.family": "serif",
        "font.serif": "Computer Modern Roman",
        "font.size": 18,
        "text.usetex": True,
        "pgf.rcfonts": False,
        "legend.loc": "upper right",
        "font.weight": "bold",  # Added bold font weight
    }
)

sns.set_palette("Set2")
sns.set_context("talk")

In [ ]:
# Load the configuration file (wich contains datasets, models, and splitting)
CFG = yaml.safe_load(open(os.path.join(DATASET_PATH, "config.yml"), "r"))

ML_MODELS: List = CFG["models"]["ML"]
GNN_MODELS: List = CFG["models"]["GNN"]["scratch"]
PRETRAINED_GNN_MODELS: List = CFG["models"]["GNN"]["pretrained"]
ALL_MODELS: List = [ML_MODELS, GNN_MODELS, PRETRAINED_GNN_MODELS]
DATASET_NAMES: List = CFG["datasets"]["TDC"]
SPLIT_TYPES: List = CFG["splitting"]


# read the results that are saved in the results folder. This is used for the visualization
results = pd.read_csv(os.path.join("classification_results", "TDC", "results.csv"))
results["model_type"] = results["model"].apply(lambda x: "Classical_ML" if x in ML_MODELS else "GNN")
metric_mapping = {"accuracy": "Accuracy", "roc_auc": "ROC-AUC", "pr_auc": "PR-AUC"}

# Scaffold

In [ ]:
dataset_category = "TDC"
dataset_name = "CYP2C19"
split_type = "scaffold"


df = pd.read_csv(os.path.join(DATASET_PATH, dataset_category, dataset_name, f"{dataset_name}_standardize.csv"))
# calculate scaffold for each molecule in the dataset
df["scaffold"] = df["smiles"].apply(get_scaffold)

In [ ]:
scaffolds = df["scaffold"].unique()

In [ ]:
# Now I want to plot 16 different moleculas that comes from 4 differnt unique scaffolds
# I will plot 3 molecules from each scaffold
scaffolds = df["scaffold"].unique()

# I will randomly select 4 scaffolds that has more than 2 molecules
i = 0
chosen_scaffolds = []
while True:
    sfc = np.random.choice(scaffolds, 1)[0]
    if df[df["scaffold"] == sfc].shape[0] > 4:
        i += 1
        chosen_scaffolds.append(sfc)
    if i == 4:
        break


# I will plot 2 molecules from each scaffold
mols = []
for scaffold in chosen_scaffolds:
    mols.append(df[df["scaffold"] == scaffold].sample(4))

allmols = pd.concat(mols)

## plot the molecules. Each columns, molecule with the same scaffold
fig, axs = plt.subplots(4, 4, figsize=(10, 10))
axs = axs.ravel()
for i, (idx, row) in enumerate(allmols.iterrows()):
    mol = Chem.MolFromSmiles(row["smiles"])
    img = Draw.MolToImage(mol, size=(200, 200))
    axs[i].imshow(img)
    axs[i].axis("off")
plt.tight_layout()
# save high quality image for slides
plt.savefig(f"assets/{dataset_name}_scaffolds.png", dpi=300)
plt.show()

In [ ]:
# plot the scaffolds in 4 rows
fig, axs = plt.subplots(4, 1, figsize=(5, 10))
axs = axs.ravel()
for i, scaffold in enumerate(chosen_scaffolds):
    mol = Chem.MolFromSmiles(scaffold)
    img = Draw.MolToImage(mol, size=(200, 200))
    axs[i].imshow(img)
    axs[i].axis("off")

plt.tight_layout()
# save high quality image for slides
plt.savefig(f"assets/{dataset_name}_scaffolds_1.png", dpi=300)
plt.show()

In [ ]:
## Calculate the number of unique scaffold in each datasetto the total size of the dataset
import yaml

with open(os.path.join(DATASET_PATH, "config.yml"), "r") as file:
    config = yaml.load(file, Loader=yaml.FullLoader)

datasets = config["datasets"]["TDC"]

scaffold_count = {}
for dataset_name in datasets:
    df = pd.read_csv(os.path.join(DATASET_PATH, "TDC", dataset_name, f"{dataset_name}_standardize.csv"))
    df["scaffold"] = df["smiles"].apply(get_scaffold)
    scaffold_count[dataset_name] = df["scaffold"].nunique() / df.shape[0]

In [ ]:
# plot the number of unique scaffold in each dataset
plt.figure(figsize=(10, 5))
plt.bar(scaffold_count.keys(), scaffold_count.values())
plt.xticks(rotation=45)
plt.ylabel("Fraction of unique scaffold", fontsize=20)
plt.tight_layout()
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.savefig("assets/scaffold_fraction.png", dpi=300)
plt.show()

# Molecular Weight

In [ ]:
# First, i calculate the molecular weight of each molecule in the dataset in the train and test split.
# Then I plot the distribution of the molecular weight in the train and test split.
split_type = "molecular_weight"
dataset_category = "TDC"
dataset_name = "HERG"

dataset_folder = os.path.join(DATASET_PATH, dataset_category, dataset_name, "split", split_type)

train = pd.read_csv(os.path.join(dataset_folder, "train_0.csv"))
test = pd.read_csv(os.path.join(dataset_folder, "test_0.csv"))

train["mw"] = train["smiles"].apply(lambda x: dm.descriptors.mw(dm.to_mol(x)))
test["mw"] = test["smiles"].apply(lambda x: dm.descriptors.mw(dm.to_mol(x)))

# Plot the histogram of the molecular weight in the train and test split
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(train["mw"], label="train", ax=ax)
sns.histplot(test["mw"], label="test", ax=ax)
ax.set_xlabel("Molecular weight", fontsize=18)
ax.set_ylabel("Count", fontsize=18)
ax.set_title(f"Molecular weight distribution in train and test {split_type} splits", fontsize=18)
ax.legend()
plt.tight_layout()
ax.grid(False)
# save high quality image for slides
# plt.savefig(f"assets/{dataset_name}_molecular_weight.png", dpi=300)
plt.show()

In [ ]:
# We want to plot 12 moelcules from the train set and then 4 molecules from the test set

train_mols = []
test_mols = []

k = 0
while k < 12:
    mol = train.sample(1)
    if mol["mw"].values[0] < 200:
        train_mols.append(mol)
        k += 1

k = 0

while k < 4:
    mol = test.sample(1)
    if mol["mw"].values[0] > 700:
        test_mols.append(mol)
        k += 1

train_mols = pd.concat(train_mols)
test_mols = pd.concat(test_mols)


fig, axs = plt.subplots(4, 4, figsize=(10, 10))
axs = axs.ravel()
for i, (idx, row) in enumerate(train_mols.iterrows()):
    mol = Chem.MolFromSmiles(row["smiles"])
    img = Draw.MolToImage(mol, size=(200, 200))
    axs[i].imshow(img)
    axs[i].axis("off")
for i, (idx, row) in enumerate(test_mols.iterrows()):
    mol = Chem.MolFromSmiles(row["smiles"])
    img = Draw.MolToImage(mol, size=(200, 200))
    axs[i + 12].imshow(img)
    axs[i + 12].axis("off")
plt.tight_layout()

# save high quality image for slides
plt.savefig(f"assets/{dataset_name}_molecular_weight_mols.png", dpi=300)

plt.show()

# Kmeans

In [ ]:
# First, I want to do kmeans splitting for the dataset.
from splito import KMeansSplit

split_type = "molecular_weight"
dataset_category = "TDC"
dataset_name = "CYP2C19"

df = pd.read_csv(os.path.join(DATASET_PATH, dataset_category, dataset_name, f"{dataset_name}_standardize.csv"))
# rn kmeans split
splitter = KMeansSplit(n_clusters=10, n_splits=1, test_size=0.2, random_state=42)

train_idx, test_idx = next(splitter.split(X=df["smiles"].values))

In [ ]:
from splito.utils import get_kmeans_clusters
from splito._distance_split_base import convert_to_default_feats_if_smiles

In [ ]:
smiles = df["smiles"].values
X, cluster_metric = convert_to_default_feats_if_smiles(smiles, "jaccard")

In [ ]:
groups = get_kmeans_clusters(
    X=X,
    n_clusters=20,
    random_state=123,
    base_metric="jaccard",
)

In [ ]:
groups = pd.DataFrame(groups, columns=["cluster"])

In [ ]:
groups

In [ ]:
## Pick forur moelcules from each cluster randomly and put the cluster ifomation in the variable mols
mols = []
for i in range(20):
    mols.append(df[groups["cluster"] == i].sample(4))

allmols = pd.concat(mols)

In [ ]:
# Then plot just for the first four clusters and 4 molecules from each cluster
sel = allmols[0:16]

fig, axs = plt.subplots(4, 4, figsize=(10, 10))
axs = axs.ravel()
for i, (idx, row) in enumerate(sel.iterrows()):
    mol = Chem.MolFromSmiles(row["smiles"])
    img = Draw.MolToImage(mol, size=(200, 200))
    axs[i].imshow(img)
    axs[i].axis("off")
plt.tight_layout()
# save high quality image for slides
plt.savefig(f"assets/{dataset_name}_kmeans.png", dpi=300)
plt.show()

# Analysis of the performance of ID vs OOD

In [ ]:
results = pd.read_csv(os.path.join("classification_results", "TDC", "results.csv"))

In [ ]:
# Just pick the chosen split from all the split in the dataframe
chosen_split = "molecular_weight"
chosen_metric = "test_accuracy"
metric_mapping = {"test_accuracy": "Accuracy", "test_roc_auc": "ROC-AUC", "test_pr_auc": "PR-AUC"}
df_selected = results[results["split"] == chosen_split]

# First, group the results by dataset. then, select ID and OOd performance columns. Then baplot ID and OOd alongside each other
fig, ax = plt.subplots(1, 1, figsize=(14, 8))
yerr = df_selected.groupby("dataset")[[f"ID_{chosen_metric}", f"OOD_{chosen_metric}"]].std()
df_selected.groupby("dataset")[[f"ID_{chosen_metric}", f"OOD_{chosen_metric}"]].mean().plot(
    kind="bar", ax=ax, yerr=yerr, capsize=5
)
ax.grid(axis="y", linestyle="--", alpha=0.6)
ax.set_xlabel("Dataset", fontsize=24)
ax.set_ylabel(metric_mapping[chosen_metric], fontsize=24)
ax.set_yticks(np.arange(0.0, 1.1, 0.1))
plt.xticks(rotation=45)
plt.tight_layout()
# save high quality image for slides
plt.savefig(f"assets/{chosen_split}_performance.png", dpi=300)
plt.show()

In [ ]:
print(
    df_selected.groupby("dataset")["ID_test_accuracy"].mean()
    - df_selected.groupby("dataset")["OOD_test_accuracy"].mean()
)
# same as above, but percentage drop
print(
    (
        df_selected.groupby("dataset")["ID_test_accuracy"].mean()
        - df_selected.groupby("dataset")["OOD_test_accuracy"].mean()
    )
    / df_selected.groupby("dataset")["ID_test_accuracy"].mean()
)

In [ ]:
df_selected.groupby("dataset")["OOD_test_accuracy"].mean()

In [ ]:
# combine results for all the datasets in the TDC folder
import yaml

dataset_category = "TDC"
with open(os.path.join(DATASET_PATH, "config.yml"), "r") as f:
    cfg = yaml.safe_load(f)
ML_MODELS = cfg["models"]["ML"]
GNN_MODELS = cfg["models"]["GNN"]["scratch"]
PRETRAINED_GNN_MODELS = cfg["models"]["GNN"]["pretrained"]
ALL_MODELS = [ML_MODELS, GNN_MODELS, PRETRAINED_GNN_MODELS]

# I want to plot the dfference bwteen ID and OOD for each model in the ALL_MODELS
diff = []
for models in ALL_MODELS:
    results = df_selected[df_selected["model"].isin(models)].copy()
    metrics = ["accuracy", "roc_auc", "pr_auc"]
    diff_models = []
    for metric in metrics:
        results[f"diff_{metric}"] = results[f"ID_test_{metric}"] - results[f"OOD_test_{metric}"]
        diff_models.append(results.groupby("model")[f"diff_{metric}"].mean())
    diff_models = pd.concat(diff_models, axis=1)
    diff.append(diff_models)

mean_df = pd.DataFrame(
    [diff[0].mean(axis=0), diff[1].mean(axis=0), diff[2].mean(axis=0)],
    index=["ML_MODELS", "GNN_MODELS", "PRETRAINED_GNN_MODELS"],
)
std_df = pd.DataFrame(
    [diff[0].std(axis=0), diff[1].std(axis=0), diff[2].std(axis=0)],
    index=["ML_MODELS", "GNN_MODELS", "PRETRAINED_GNN_MODELS"],
)

In [ ]:
# plot the diff_roc_auc for ML_MODELS, GNN_MODELS, PRETRAINED_GNN_MODELS separately alongside each other
fig, ax = plt.subplots(figsize=(8, 8), nrows=1, ncols=1)
mean_df["diff_accuracy"].plot(kind="bar", ax=ax, yerr=std_df["diff_accuracy"], capsize=5)
ax.set_xlabel("Model", fontsize=20)
ax.set_ylabel("Accuracy Difference", fontsize=20)
plt.xticks(rotation=45)
ax.grid(True, axis="y", linestyle="--")
plt.tight_layout()
# save high quality image for slides
plt.savefig("assets/model_performance_agg_mw.png", dpi=300)
plt.show()

In [ ]:
# I want to plot the dfference bwteen ID and OOD for each model in the ALL_MODELS for each dataset separately
diff = []
metrics = "accuracy"
percentage = True
for models in ALL_MODELS:
    results = df_selected[df_selected["model"].isin(models)].copy()
    diff_models = []
    if percentage:
        results[f"diff_{metrics}"] = (
            (results[f"ID_test_{metrics}"] - results[f"OOD_test_{metrics}"]) / results[f"ID_test_{metrics}"] * 100
        )
    else:
        results[f"diff_{metrics}"] = results[f"ID_test_{metrics}"] - results[f"OOD_test_{metrics}"]
    diff_models.append(results.groupby(["model", "dataset"])[f"diff_{metrics}"].mean())
    diff_models = pd.concat(diff_models, axis=1)
    diff.append(diff_models)

# for all the datasets, plot the diff_roc_auc for ML_MODELS, GNN_MODELS, PRETRAINED_GNN_MODELS separately alongside each other
a = pd.DataFrame()
for i in range(3):
    a = pd.concat([a, diff[i].groupby("dataset").mean()], axis=1)
a.columns = ["ML_MODELS", "GNN_MODELS", "PRETRAINED_GNN_MODELS"]

In [ ]:
# plot the diff_roc_auc for ML_MODELS, GNN_MODELS, PRETRAINED_GNN_MODELS separately alongside each other
fig, ax = plt.subplots(figsize=(14, 8), nrows=1, ncols=1)
a.plot(kind="bar", ax=ax)
ax.set_xlabel("Dataset", fontsize=20)
ax.set_ylabel("Accuracy Difference \% ", fontsize=20)
ax.grid(True, axis="y", linestyle="--")
plt.xticks(rotation=45)
plt.tight_layout()
# save high quality image for slides
plt.savefig(f"assets/{chosen_split}_performance_separated.png", dpi=300)
plt.show()

## Molecular weight analysis

In [ ]:
# First calculate molecular weight for train and test set. Then, break down the test set to three different bins based on molecular weight
# Then, find out the prediction on each bin, and then calculate accuracy for each bin

import yaml

cfg = yaml.safe_load(open(os.path.join(DATASET_PATH, "config.yml"), "r"))

dataset_category = "TDC"
dataset_names = cfg["datasets"][dataset_category]
split_type = "molecular_weight"
model_names = cfg["models"]["ML"] + cfg["models"]["GNN"]["scratch"] + cfg["models"]["GNN"]["pretrained"]
GNN_MODELS = cfg["models"]["GNN"]["scratch"]
PRETRAINED_GNN_MODELS = cfg["models"]["GNN"]["pretrained"]
ML_MODELS = cfg["models"]["ML"]
ALL_MODELS = [ML_MODELS, GNN_MODELS, PRETRAINED_GNN_MODELS]

index = 0
accuracy_list = []
for dataset_name in dataset_names:
    for model_name in model_names:
        SPLIT_PATH = os.path.join(DATASET_PATH, dataset_category, dataset_name, "split")
        df = pd.read_csv(
            os.path.join(
                "classification_results",
                dataset_category,
                dataset_name,
                split_type,
                model_name,
                str(index + 1),
                "prediction.csv",
            )
        )
        df1 = pd.read_csv(os.path.join(SPLIT_PATH, split_type, f"test_{index}.csv"))

        # Compute molecular weight
        df1["mw"] = df1["smiles"].apply(lambda x: dm.descriptors.mw(dm.to_mol(x)))

        df["prediction"] = df["label"] > 0.5
        df["label"] = df1["label"]

        # Break down the test set to three different bins based on molecular weight
        bins_range = [min(df1["mw"]) - 0.01, df1["mw"].quantile(0.4), df1["mw"].quantile(0.8), max(df1["mw"] + 0.01)]

        # Find out the prediction on each bin
        df["mw"] = df1["mw"]
        df["bin"] = pd.cut(df["mw"], bins=bins_range, labels=["low", "medium", "high"])

        # Calculate accuracy for each bin
        accuracy = df.groupby("bin").apply(lambda x: (x["label"] == x["prediction"]).mean())
        print(f"Accuracy for split {index}")
        accuracy_list.append(accuracy)

In [ ]:
accuracy_list = pd.concat(accuracy_list, axis=1)

In [ ]:
# column name for accuracy list which is dataset_name + model_name
accuracy_list.columns = [f"{dataset_name}_{model_name}" for dataset_name in dataset_names for model_name in model_names]

In [ ]:
# For each datset,mean GNN and ML models and plot for each bin
fig, axs = plt.subplots(2, 4, figsize=(15, 8))
axs = axs.ravel()
for i, dataset_name in enumerate(dataset_names):
    accuracy_list[[f"{dataset_name}_{model}" for model in GNN_MODELS]].mean(axis=1).plot(
        kind="bar",
        ax=axs[i],
        yerr=accuracy_list[[f"{dataset_name}_{model}" for model in GNN_MODELS]].std(axis=1),
        capsize=5,
    )
    axs[i].set_title(f"{dataset_name}")
    axs[i].set_ylabel("Accuracy")
    axs[i].set_xlabel("Molecular weight")
    axs[i].set_ylim(0, 0.85)
    axs[i].grid(True, axis="y", linestyle="--")
plt.tight_layout()
# save high quality image for slides
plt.show()

In [ ]:
# same plot for ML models
fig, axs = plt.subplots(2, 4, figsize=(15, 8))
axs = axs.ravel()
for i, dataset_name in enumerate(dataset_names):
    accuracy_list[[f"{dataset_name}_{model}" for model in ML_MODELS]].mean(axis=1).plot(
        kind="bar",
        ax=axs[i],
        yerr=accuracy_list[[f"{dataset_name}_{model}" for model in ML_MODELS]].std(axis=1),
        capsize=5,
    )
    axs[i].set_title(f"{dataset_name}")
    axs[i].set_xlabel("Molecular weight")
    axs[i].set_ylabel("Accuracy")
    axs[i].set_ylim(0, 0.85)
    axs[i].grid(True, axis="y", linestyle="--")
plt.tight_layout()
# save high quality image for slides
plt.savefig(f"assets/{split_type}_performance_bins.png", dpi=300)
plt.show()

In [ ]:
# Can you plot the same as abov, but with scatter plot which connect the dots with dashed line
# Each plot, ML and GNN models with two differnt clor for one dataset
fig, axs = plt.subplots(2, 4, figsize=(15, 8))
axs = axs.ravel()
for i, dataset_name in enumerate(dataset_names):
    accuracy_list[[f"{dataset_name}_{model}" for model in ML_MODELS]].mean(axis=1).plot(
        ax=axs[i], linestyle="--", marker="o"
    )
    accuracy_list[[f"{dataset_name}_{model}" for model in GNN_MODELS]].mean(axis=1).plot(
        ax=axs[i], linestyle="--", marker="o"
    )
    axs[i].set_title(f"{dataset_name}")
    axs[i].set_xlabel("Molecular weight")
    axs[i].set_ylabel("Accuracy")
    axs[i].set_ylim(0.2, 0.85)
    axs[i].grid(True, axis="y", linestyle="--")
    # add legend
    axs[i].legend(["ML", "GNN"], loc="lower right")
plt.tight_layout()
# save high quality image for slides
plt.show()

In [ ]:
# just look at more carefully at the bin and accuracy for one dataset
# Break down the test set to three different bins based on molecular weight
index = 0
dataset_name = "CYP1A2"
SPLIT_PATH = os.path.join(DATASET_PATH, dataset_category, dataset_name, "split")
df = pd.read_csv(
    os.path.join(
        "classification_results",
        dataset_category,
        dataset_name,
        split_type,
        model_name,
        str(index + 1),
        "prediction.csv",
    )
)
df1 = pd.read_csv(os.path.join(SPLIT_PATH, split_type, f"test_{index}.csv"))
# Compute molecular weight
df1["mw"] = df1["smiles"].apply(lambda x: dm.descriptors.mw(dm.to_mol(x)))

df["prediction"] = df["label"] > 0.5
df["label"] = df1["label"]
bins_range = [min(df1["mw"]) - 0.0001, df1["mw"].quantile(0.4), df1["mw"].quantile(0.8), max(df1["mw"] + 0.0001)]

# Find out the prediction on each bin
df["mw"] = df1["mw"]
df["bin"] = pd.cut(df["mw"], bins=bins_range, labels=["low", "medium", "high"])

In [ ]:
# ratio of label 1 in each bin
df.groupby("bin")["label"].mean()

In [ ]:
df["label"].mean()

# TMD Distance

In [ ]:
# First read the numpy array that contains the TMD distances
dataset_name = "HIV"

TMD_MAT = np.load(os.path.join(DATASET_PATH, "TDC", dataset_name, "TMD_distance.npy"))

In [ ]:
# Load the hav data to know the smiles corresponding to each row in the TMD distance matrix

df = pd.read_csv(os.path.join(DATASET_PATH, "TDC", dataset_name, f"{dataset_name}_standardize.csv"))

In [ ]:
# Load particular train and test split
split_type = "molecular_weight"
split = 0
train = pd.read_csv(os.path.join(DATASET_PATH, "TDC", dataset_name, "split", split_type, f"train_{split}.csv"))
test = pd.read_csv(os.path.join(DATASET_PATH, "TDC", dataset_name, "split", split_type, f"test_{split}.csv"))

# Compute molecular weight
test["mw"] = test["smiles"].apply(lambda x: dm.descriptors.mw(dm.to_mol(x)))


# Break down the test set to three different bins based on molecular weight
bins_range = [min(test["mw"]) - 0.0001, test["mw"].quantile(0.4), test["mw"].quantile(0.8), max(test["mw"] + 0.0001)]

# Find out the prediction on each bin
test["bin"] = pd.cut(test["mw"], bins=bins_range, labels=["low", "medium", "high"])

# Find the indices in the original dataset that correwsponds to the smiles in the train and test set
train_indices = []
test_indices = []
for idx, row in train.iterrows():
    train_indices.append(df[df["smiles"] == row["smiles"]].index[0])
for idx, row in test.iterrows():
    test_indices.append(df[df["smiles"] == row["smiles"]].index[0])

In [ ]:
# Now select the rows from train indices and column from the test indices in the TMD distance matrix
TMD_subset = TMD_MAT[train_indices, :][:, test_indices]

In [ ]:
# for each column in the TMD distance matrix, find the minimum value and the index of the minimum value
print(TMD_subset.min(axis=0).mean())
print(TMD_subset.min(axis=0).std())
print(np.median(TMD_subset.min(axis=0)))

In [ ]:
# calculate test indices for each bin, and then calculate the TMD distance for each bin
bin_indices = []
for bin_name in ["low", "medium", "high"]:
    bin_indices.append(test[test["bin"] == bin_name].index)

bin_TMD = []
# calculate the minimum TMD distance for each bin and then Median of the minimum TMD distance
for indices in bin_indices:
    bin_TMD.append(np.median(TMD_subset[:, indices].min(axis=0)))

In [ ]:
bin_TMD

In [ ]:
TMD_subset[:, bin_indices[0]].shape

In [ ]:
TMD_subset.shape

In [ ]:
np.array(bin_TMD).mean()

In [ ]:
# Just write the whole process into a function
def calculate_TMD_distance(dataset_name, split_type, TMD_matrix, index=0):
    df = pd.read_csv(os.path.join(DATASET_PATH, "TDC", dataset_name, f"{dataset_name}_standardize.csv"))
    train = pd.read_csv(os.path.join(DATASET_PATH, "TDC", dataset_name, "split", split_type, f"train_{index}.csv"))
    test = pd.read_csv(os.path.join(DATASET_PATH, "TDC", dataset_name, "split", split_type, f"test_{index}.csv"))
    test["mw"] = test["smiles"].apply(lambda x: dm.descriptors.mw(dm.to_mol(x)))
    bins_range = [
        min(test["mw"]) - 0.0001,
        test["mw"].quantile(0.4),
        test["mw"].quantile(0.8),
        max(test["mw"] + 0.0001),
    ]
    test["bin"] = pd.cut(test["mw"], bins=bins_range, labels=["low", "medium", "high"])
    train_indices = []
    test_indices = []
    for idx, row in train.iterrows():
        train_indices.append(df[df["smiles"] == row["smiles"]].index[0])
    for idx, row in test.iterrows():
        test_indices.append(df[df["smiles"] == row["smiles"]].index[0])
    TMD_subset = TMD_matrix[train_indices, :][:, test_indices]
    bin_indices = []
    for bin_name in ["low", "medium", "high"]:
        bin_indices.append(test[test["bin"] == bin_name].index)
    bin_TMD = []
    for indices in bin_indices:
        bin_TMD.append(TMD_subset[:, indices].min(axis=0).mean())
    return bin_TMD

In [ ]:
CYP2D6_dist = calculate_TMD_distance("CYP2D6", "molecular_weight", TMD_MAT, index=0)

In [ ]:
HIV_dist = calculate_TMD_distance("HIV", "molecular_weight", TMD_MAT, index=0)

In [ ]:
TMD_list = [CYP2D6_dist, HIV_dist]

In [ ]:
dataset_names = ["CYP2D6", "HIV"]
acc = []
for i, dataset_name in enumerate(dataset_names):
    acc.append(accuracy_list[[f"{dataset_name}_{model}" for model in GNN_MODELS]].mean(axis=1))

In [ ]:
df

In [ ]:
# plot the accuracy for each dataset barplot
fig, axs = plt.subplots(1, 2, figsize=(15, 8))
axs = axs.ravel()
for i, dataset_name in enumerate(dataset_names):
    acc[i].plot(
        kind="bar",
        ax=axs[i],
        yerr=accuracy_list[[f"{dataset_name}_{model}" for model in GNN_MODELS]].std(axis=1),
        capsize=5,
    )
    axs[i].set_title(f"{dataset_name}")
    axs[i].set_ylabel("Accuracy", fontsize=20)
    axs[i].set_xlabel("Molecular weight", fontsize=20)
    axs[i].set_ylim(0.2, 0.85)
    axs[i].grid(True, axis="y", linestyle="--")
plt.tight_layout()
# save high quality image for slides
plt.savefig(f"assets/{split_type}_performance_bins_analysis.png", dpi=300)
plt.show()

In [ ]:
# plot the TMD list for each dataset barplot
fig, axs = plt.subplots(1, 2, figsize=(15, 8))
axs = axs.ravel()
for i, dataset_name in enumerate(dataset_names):
    pd.Series(TMD_list[i]).plot(kind="bar", ax=axs[i], color="orange")
    axs[i].set_title(f"{dataset_name}")
    axs[i].set_ylabel("TMD distance", fontsize=20)
    axs[i].set_xlabel("Molecular weight", fontsize=20)
    axs[i].grid(True, axis="y", linestyle="--")
    axs[i].set_ylim(0, 1200)
    axs[i].set_xticklabels(["low", "medium", "high"], rotation=90, fontsize=20)
    # plot the libe for average of the TMD distance
    axs[i].axhline(y=np.mean(TMD_list[i]), color="r", linestyle="--")
plt.tight_layout()
# save high quality image for slides
plt.savefig(f"assets/{split_type}_TMD_distance_analysis.png", dpi=300)
plt.show()

In [ ]:
TMD_list

# Plot performance of ID vs OOD

In [ ]:
# First groupby by the split. Then box plot the differnce between ID_test_roc_auc and OOD_test_roc_auc for each split
# Create a single figure with two subplots
# box plots with the difference between ID and OOD test aggregated by dataset
# heatmap with the difference between ID and OOD test aggregated by dataset and split


# Metric setup
metric = "roc_auc"
metric_mapping = {"accuracy": "Accuracy", "roc_auc": "ROC-AUC", "pr_auc": "PR-AUC"}
perc = False
save = False

# Compute difference between ID and OOD test performance
results["difference"] = results[f"ID_test_{metric}"] - results[f"OOD_test_{metric}"]

# Create figure with 2 wide subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8), gridspec_kw={"wspace": 0.25})


# Bold tick labels using LaTeX
def bold_ticks(ax):
    ax.set_xticklabels([r"\textbf{" + label.get_text() + "}" for label in ax.get_xticklabels()], fontsize=18)
    ax.set_yticklabels([r"\textbf{" + label.get_text() + "}" for label in ax.get_yticklabels()], fontsize=18)


# --- Subplot (a): Boxplot ---
sns.boxplot(x="split", y="difference", data=results, ax=ax1, hue="split", showfliers=True, palette="Set2")
# ax1.set_xlabel(r"\textbf{Split Type}", fontsize=22)
ax1.set_ylabel(r"\textbf{$\Delta$ " + metric_mapping[metric] + r"}", fontsize=22)
ax1.set_title(r"\textbf{Distribution of ID-OOD $\Delta$ " + metric_mapping[metric] + r"}", fontsize=24)
ax1.tick_params(axis="x", labelsize=16, rotation=60)
ax1.tick_params(axis="y", labelsize=16)
for tick in ax1.get_xticklabels():
    tick.set_fontsize(16)
    tick.set_rotation(60)
    tick.set_ha("right")
    tick.set_fontweight("bold")
for tick in ax1.get_yticklabels():
    tick.set_fontsize(16)
    tick.set_fontweight("bold")
ax1.grid(axis="y", linestyle="--", alpha=0.6)
ax1.legend().remove()
ax1.text(-0.15, 1.05, r"\textbf{(a)}", transform=ax1.transAxes, fontsize=22)

# --- Subplot (b): Heatmap ---
dataset_names = results["dataset"].unique()
split_types = SPLIT_TYPES
df = pd.DataFrame(index=dataset_names, columns=split_types)

for dataset in dataset_names:
    for split in split_types:
        id_val = results[(results["dataset"] == dataset) & (results["split"] == split)][f"ID_test_{metric}"].mean()
        ood_val = results[(results["dataset"] == dataset) & (results["split"] == split)][f"OOD_test_{metric}"].mean()
        df.loc[dataset, split] = id_val - ood_val if not perc else (id_val - ood_val) / id_val * 100

df = df.astype(float)

sns.heatmap(df, ax=ax2, cmap="coolwarm", annot=True, fmt=".3f", vmin=0.0, vmax=0.2, cbar_kws={"shrink": 0.8})
# ax2.set_xlabel(r"\textbf{Split Type}", fontsize=22)
ax2.set_ylabel(r"\textbf{Datasets}", fontsize=22)
ax2.set_title(r"\textbf{Mean ID-OOD $\Delta$ " + metric_mapping[metric] + r"}", fontsize=24)
ax2.tick_params(axis="x", labelsize=16, rotation=60)
ax2.tick_params(axis="y", labelsize=16)
for tick in ax2.get_xticklabels():
    tick.set_fontsize(16)
    tick.set_rotation(60)
    tick.set_ha("right")
    tick.set_fontweight("bold")
for tick in ax2.get_yticklabels():
    tick.set_fontsize(16)
    tick.set_fontweight("bold")
ax2.text(-0.15, 1.05, r"\textbf{(b)}", transform=ax2.transAxes, fontsize=22)

bold_ticks(ax1)
bold_ticks(ax2)

# Save if needed
if save:
    plt.savefig("assets/box_heatmap_id_ood_comparison_roc_auc_poster.pdf", bbox_inches="tight")
    plt.savefig("assets/box_heatmap_id_ood_comparison_roc_auc_poster.png", bbox_inches="tight", dpi=300)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Setup matplotlib
import matplotlib

matplotlib.rcParams.update(
    {
        "pgf.texsystem": "pdflatex",
        "font.family": "serif",
        "font.serif": "Computer Modern Roman",
        "font.size": 16,
        "text.usetex": True,
        "pgf.rcfonts": False,
        "legend.loc": "upper right",
    }
)

sns.set_palette("Set2")
sns.set_context("talk")

# Create grouped bar plot for ML vs GNN model differences
save = True

results["model_type"] = results["model"].apply(lambda x: "Classical_ML" if x in ML_MODELS else "GNN")

fig = plt.figure(figsize=(16, 12))  # Wider and taller for poster
gs = gridspec.GridSpec(
    2, 1, height_ratios=[1, 1], hspace=0.75, wspace=0.35, figure=fig
)  # Increased hspace for more gap


# First row: one wide subplot
gs1 = gridspec.GridSpecFromSubplotSpec(1, 1, subplot_spec=gs[0])
ax1 = plt.subplot(gs1[0, 0])

# Second and third rows: 4x2 grid (8 subplots)
gs2 = gridspec.GridSpecFromSubplotSpec(2, 4, subplot_spec=gs[1])
axes = [plt.subplot(gs2[i, j]) for i in range(2) for j in range(4)]

df = results.groupby(["split", "model_type", "dataset"])["difference"].mean().reset_index()
df["split"] = pd.Categorical(df["split"], SPLIT_TYPES)

# Main plot
g = sns.barplot(x="split", y="difference", hue="model_type", data=df, ax=ax1, legend=True)
ax1.set_xlabel("")  # No x-label on main plot
ax1.set_ylabel(r"\textbf{$\Delta$ " + metric_mapping[metric] + "}", fontsize=20)
ax1.set_title(r"\textbf{Difference between ID and OOD test " + metric_mapping[metric] + "}", fontsize=24, pad=20)

# x-ticks
xticklabels = [label.get_text() for label in ax1.get_xticklabels()]
ax1.set_xticklabels([r"\textbf{" + label + "}" for label in xticklabels], rotation=15, ha="right", fontsize=16)
yticklabels = [label.get_text() for label in ax1.get_yticklabels()]
ax1.set_yticklabels([r"\textbf{" + label + "}" for label in yticklabels], fontsize=16)

# Grid and legend
ax1.grid(axis="y", linestyle="--", alpha=0.6)
legend = ax1.legend(loc="upper left", bbox_to_anchor=(1, 1), fontsize=14, title=r"\textbf{Model Type}")
for text in legend.get_texts():
    text.set_fontsize(14)
    text.set_text(r"\textbf{" + text.get_text() + "}")

# Subplots
for i, dataset in enumerate(dataset_names):
    df_dataset = df[df["dataset"] == dataset]
    g = sns.barplot(x="split", y="difference", hue="model_type", data=df_dataset, ax=axes[i])

    axes[i].set_title(r"\textbf{" + dataset + "}", fontsize=18, pad=10)
    axes[i].set_xlabel("")
    if i % 4 == 0:
        axes[i].set_ylabel(r"\textbf{$\Delta$ " + metric_mapping[metric] + "}", fontsize=14)
    else:
        axes[i].set_ylabel("")

    # x-tick formatting
    if i >= 4:
        xticklabels = [label.get_text() for label in axes[i].get_xticklabels()]
        axes[i].set_xticklabels(
            [r"\textbf{" + label + "}" for label in xticklabels], rotation=60, ha="right", fontsize=12
        )
    else:
        axes[i].set_xticklabels([])

    # y-tick formatting
    # yticklabels = [label.get_text() for label in axes[i].get_yticklabels()]
    # axes[i].set_yticklabels([r'\textbf{' + label + '}' for label in yticklabels], fontsize=10)

    axes[i].tick_params(axis="both", labelsize=14)
    axes[i].grid(axis="y", linestyle="--", alpha=0.5)
    g.legend().remove()
    axes[i].set_ylim(-0.10, 0.20)
    axes[i].set_yticks(np.arange(-0.10, 0.21, 0.05))

# Subplot labels
ax1.text(-0.1, 1.05, r"\textbf{(a)}", transform=ax1.transAxes, fontsize=20)
axes[0].text(-0.55, 1.25, r"\textbf{(b)}", transform=axes[0].transAxes, fontsize=20)

plt.subplots_adjust(left=0.07, right=0.95, top=0.93, bottom=0.08, hspace=0.75, wspace=0.35)  # Increased hspace

if save:
    plt.savefig("assets/figures/grouped_barplot_ml_gnn_difference_poster.pdf", bbox_inches="tight")
    plt.savefig("assets/figures/grouped_barplot_ml_gnn_difference_poster.png", bbox_inches="tight", dpi=600)  # High-res

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import numpy as np
import matplotlib as mpl

save = True
metric = "roc_auc"

# Update rcParams for LaTeX rendering with proper bold support
mpl.rcParams.update(
    {
        "text.usetex": True,
        "pgf.texsystem": "pdflatex",
        "font.family": "serif",
        "font.serif": ["Computer Modern Roman"],
        "font.size": 14,
        "pgf.rcfonts": False,
        "legend.loc": "upper right",
    }
)


def make_regplot(x, y, ax, color="#66c2a5"):
    sns.regplot(
        x=x,
        y=y,
        ax=ax,
        scatter_kws={"alpha": 0.5, "s": 40},
        line_kws={"color": "red", "linewidth": 2},
        ci=95,
        color=color,
    )

    corr = pearsonr(x, y)[0]
    props = dict(boxstyle="round", facecolor="white", alpha=0.8, edgecolor="gray", linewidth=0.5)
    ax.text(
        0.05,
        0.95,
        rf"\textbf{{r = {corr:.2f}}}",
        transform=ax.transAxes,
        fontsize=14,
        verticalalignment="top",
        bbox=props,
    )


def format_axis(ax):
    lims = [0.5, 0.95]
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.plot(lims, lims, "--", alpha=1, color="gray", linewidth=2)
    ax.set_xticks(np.arange(0.5, 1.0, 0.1))
    ax.set_yticks(np.arange(0.5, 1.0, 0.1))
    # Bold tick labels
    ax.set_xticklabels([rf"\textbf{{{tick:.1f}}}" for tick in np.arange(0.5, 1.0, 0.1)], fontsize=14)
    ax.set_yticklabels([rf"\textbf{{{tick:.1f}}}" for tick in np.arange(0.5, 1.0, 0.1)], fontsize=14)


fig = plt.figure(figsize=(16, 12))
# gs = gridspec.GridSpec(5, 4, height_ratios=[2, 1, 1, 1, 1], hspace=1.0, wspace=0.3)
gs = gridspec.GridSpec(2, 1, height_ratios=[1, 4], hspace=0.4, wspace=0.3)  # new

fig.suptitle(rf"\textbf{{ID vs OOD Performance Comparison ({metric_mapping[metric]})}}", fontsize=24, y=0.95)

gs1 = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=gs[0], wspace=0.15)
ax1 = plt.subplot(gs1[0, 0])
ax2 = plt.subplot(gs1[0, 1])

gs2 = gridspec.GridSpecFromSubplotSpec(4, 4, subplot_spec=gs[1], hspace=1, wspace=0.3)

ml_axes = [plt.subplot(gs2[i, j]) for i in range(4) for j in range(2)]
gnn_axes = [plt.subplot(gs2[i, j]) for i in range(4) for j in range(2, 4)]


# ax1 = plt.subplot(gs[0, 0:2])
# ax2 = plt.subplot(gs[0, 2:4])

# ml_axes = [plt.subplot(gs[i, j]) for i in range(1, 5) for j in range(2)]
# gnn_axes = [plt.subplot(gs[i, j]) for i in range(1, 5) for j in range(2, 4)]

ML_result = results[results["model_type"] == "Classical_ML"]
GNN_result = results[results["model_type"] == "GNN"]

make_regplot(
    ML_result[f"ID_test_{metric}"],
    ML_result[f"OOD_test_{metric}"],
    ax1,
)
make_regplot(GNN_result[f"ID_test_{metric}"], GNN_result[f"OOD_test_{metric}"], ax2, color="#8da0cb")

for ax, title in [(ax1, "Classical ML Models"), (ax2, "GNN Models")]:
    ax.set_title(rf"\textbf{{{title}}}", fontsize=20, pad=10)
    ax.set_xlabel(rf"\textbf{{ID {metric_mapping[metric]}}}", fontsize=15)
    ax.set_ylabel(rf"\textbf{{OOD {metric_mapping[metric]}}}", fontsize=15 if ax == ax1 else 0)
    ax.tick_params(axis="both", labelsize=15)
    format_axis(ax)

for i, split in enumerate(SPLIT_TYPES):
    ML_split = ML_result[ML_result["split"] == split]
    GNN_split = GNN_result[GNN_result["split"] == split]

    make_regplot(ML_split[f"ID_test_{metric}"], ML_split[f"OOD_test_{metric}"], ml_axes[i])
    make_regplot(GNN_split[f"ID_test_{metric}"], GNN_split[f"OOD_test_{metric}"], gnn_axes[i], color="#8da0cb")

    for ax in [ml_axes[i], gnn_axes[i]]:
        ax.set_title(rf"\textbf{{{split}}}", fontsize=18)
        format_axis(ax)
        ax.tick_params(axis="both", labelsize=14)

        if ax == ml_axes[i] and i % 2 == 0:
            ax.set_ylabel(rf"\textbf{{OOD {metric_mapping[metric]}}}", fontsize=12)
        else:
            ax.set_ylabel("")

        if i >= len(SPLIT_TYPES) - 2:
            ax.set_xlabel(rf"\textbf{{ID {metric_mapping[metric]}}}", fontsize=12)
        else:
            ax.set_xlabel("")

ax1.text(-0.15, 1.2, r"\textbf{(a)}", transform=ax1.transAxes, fontsize=20)
ml_axes[0].text(-0.35, 1.6, r"\textbf{(b)}", transform=ml_axes[0].transAxes, fontsize=20)

if save:
    plt.savefig("assets/figures/regplot_with_categories_poster.pdf", bbox_inches="tight")
    plt.savefig("assets/figures/regplot_with_categories_poster.png", bbox_inches="tight", dpi=300)
plt.show()